In [1]:
import pandas as pd
from vrpy import VehicleRoutingProblem
import networkx as nx
from tqdm import tqdm 
import json
import warnings
import logging
import time
import numpy as np

warnings.filterwarnings("ignore")
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)


In [2]:
df_grids = pd.read_csv('../../results/utils/quadrados_clusterizados_todos_os_clusters.csv').drop('Unnamed: 0', axis=1).drop(['maxLat','minLat','maxLong','minLong'], axis=1)
df_grids['Grid'] = df_grids['Grid'].apply(int)
df_grids = df_grids.astype({'candidate':'string', 'Grid': 'string'})

df_demands = pd.read_csv('../../results/utils/df_demanda_quadrados.csv').drop('Unnamed: 0', axis=1)
df_demands['Grid'] = df_demands['Grid'].apply(int)
df_demands = df_demands.astype({'Date':'string', 'Grid': 'string'})

df_dist_dcs = pd.read_csv('../../results/utils/df_dist_dcs.csv').drop('Unnamed: 0', axis=1)
df_dist_dcs['grid'] = df_dist_dcs['grid'].apply(int)
df_dist_dcs = df_dist_dcs.astype({'grid': 'string', 'CD': 'string'})

df_dist_grids = pd.read_csv('../../results/utils/df_dist_grids.csv').drop('Unnamed: 0', axis=1)
df_dist_grids['grid1'] = df_dist_grids['grid1'].apply(int)
df_dist_grids['grid2'] = df_dist_grids['grid2'].apply(int)
df_dist_grids = df_dist_grids.astype({'grid1': 'string', 'grid2': 'string'})

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Lenovo\\OneDrive\\Documentos\\André Luis\\1. UFMG\\TCC\\TCC\\Resultados\\Utils\\quadrados_clusterizados_todos_os_clusters.csv'

In [3]:
# define iteration variables
num = df_grids['n_clusters'].sort_values(ascending=False).unique()
instances = df_demands.drop(['Grid', 'Date'], axis=1).columns
days = df_demands['Date'].unique()

# define constraint variables
speed = {'carro': 32, 'moto': 40}
loading_time = 0.08
cost_per_km = {'carro': 1.5, 'moto': 1}
load_capacity=60 
duration=6
fixed_cost=80

In [4]:
def add_result(n, i, d, c, prob):
    
    result = {}
    result['n_clusters'] = [n]
    result['instance'] = [i]
    result['day'] = [d]
    result['cluster'] = [c]
    result['num_routes'] = len(prob.best_routes)
    result['avg_route_duration'] = sum(list(prob.best_routes_duration.values()))/len(prob.best_routes)
    result['total_route_cost'] = sum(list(prob.best_routes_cost.values()))
    result['max_route_cost'] = max(list(prob.best_routes_cost.values()))
    
    j=0
    total_cost_val = 0
    route_count = 0
    freight_per_route = []
    for route in list(prob.best_routes.values()):
        count = len(route) - 2
        custo = list(prob.best_routes_cost.values())[j]
        total_cost_val = total_cost_val + custo
        route_count = route_count + count
        freight_per_route.append(custo/count)
        j = j+1

    result['avg_freight'] = sum(freight_per_route)/len(freight_per_route)
    result['max_freight'] = max(freight_per_route)
    result['min_freight'] = min(freight_per_route)
    result['clients'] = route_count

    return result

In [ ]:
results_df = pd.DataFrame(columns=['n_clusters', 'instance', 'day', 'cluster', 'num_routes', 'total_route_cost', 'max_route_cost', 
                                        'avg_freight', 'clients', 'max_freight', 'min_freight', 'avg_route_duration'])

# iterate over number of clusters (1, 3, 5 and 7)
for n in num:
    if n == 1:
        break
    grids = df_grids[df_grids['n_clusters'] == n]

    print(n)
    # iterate over each instance
    for i in instances:
        demands = df_demands[['Grid', 'Date', str(i)]]
        
        group = demands[['Date', str(i)]].groupby(by='Date').sum().reset_index()
        sort = group.sort_values(by=str(i), ascending=False).reset_index()
        dataMax = sort['Date'][0:30]

        # iterate over each day
        for d in dataMax:
            # main dataframe with demands per grid cell for the current cluster
            df_demands_cluster = demands[demands['Date'] == str(d)].merge(grids, on='Grid', how='left')
            df_demands_cluster = df_demands_cluster[df_demands_cluster[i] > 0]
            
            # iterate over each cluster
            clusters = df_demands_cluster['cluster'].unique()
            for c in clusters:
                # define data
                # main dataframe with demands per grid cell for the specific cluster
                df = df_demands_cluster.loc[df_demands_cluster['cluster'] == c].reset_index().drop(['index'], axis=1)
                
                # dataframe with distances between grid cells and distribution centers 
                dists_dcs = df_dist_dcs[(df_dist_dcs['CD'].isin(df['candidate']))]
                dists_dcs = dists_dcs[(dists_dcs['grid'].isin(df['Grid']))]
                
                # dataframe with distances between grid cells
                dists_grids = df_dist_grids[(df_dist_grids['grid1'].isin(df['Grid']))]
                dists_grids = dists_grids[(dists_grids['grid2'].isin(df['Grid']))]

                # convert distance to cost and time
                df_cost = dists_grids
                df_cost['Custo'] = df_cost['Distance'] * cost_per_km['carro']
                df_cost['Tempo'] = df_cost['Distance']/speed['carro'] + loading_time

                # convert column to dictionary
                df_cost['attributes'] = '{"cost": ' + df_cost['Custo'].apply(str) + ', "time": ' + df_cost['Tempo'].apply(str) + '}'
                df_cost['attributes'] = df_cost['attributes'].apply(json.loads)
                
                # declare graph and create nodes
                G = nx.from_pandas_edgelist(df_cost, 'grid1', 'grid2', create_using=nx.DiGraph())

                # create edges and add costs from cost matrix
                G.add_edges_from(list(df_cost[['grid1', 'grid2', 'attributes']].itertuples(index=False, name=None)))
                
                # create edges with costs including Source and Sink (cluster distribution center)
                for k, linha in dists_dcs.iterrows():
                    G.add_edge("Source", linha['grid'],cost=linha['Distance'])
                    G.add_edge(linha['grid'], "Sink",cost=linha['Distance'])
                    
                # set demand for each client
                for o, r in df.iterrows():
                    nx.set_node_attributes(G, {r['Grid']: r[str(i)]}, 'demand')

                # solve the VRP using the graph
                prob = VehicleRoutingProblem(G, mixed_fleet=False, load_capacity=load_capacity, duration=duration, fixed_cost=fixed_cost)
                prob.solve(time_limit=10, heuristic_only=True, cspy=False)
                
                # save results
                result = add_result(n, i, d, c, prob)
                results_df = results_df.append(pd.DataFrame(result)) 
                
results_df = results_df.reset_index().drop(columns='index', index=1)
results_df['fixed_cost'] = 250 * results_df['n_clusters']
results_df['total_cost'] = results_df['fixed_cost'] + results_df['total_route_cost']
results_df['avg_cost_per_route'] = results_df['total_cost']/results_df['num_routes']
results_df['max_cost_dispersion'] = results_df['max_route_cost'] - results_df['avg_route_duration']

In [10]:
results = results_df

In [11]:
results.to_csv('../../results/results_df.csv')